In [ ]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
import logging


#-- 1� JOIN ACORDOS X PARCELAS
def join_parcelas(bi_acordos_promessas, bi_parcelas):
    df_acordos_promessas = (bi_acordos_promessas.join(bi_parcelas,
                                  bi_acordos_promessas['COD_PARCELAMENTO'] == bi_parcelas['COD_PARCELAMENTO_PAR'], "left"))
      
    
    return (df_acordos_promessas
            .drop(bi_parcelas['COD_CONTRATO'])
            .drop(bi_acordos_promessas['DAT_PROCESSAMENTO'])
            .drop(bi_acordos_promessas['COD_CLIENTE'])
           )
        
#-- 2� EMPACOTAMENTO
def join_originais(df_acordos_promessas, bi_originais):
    
    bi_originais = (bi_originais.filter(
                                (f.col('NOM_NATUREZA_TITULO') == 'MENSALIDADE') &
                                (f.col('IND_NATUREZA_REMATRICULA') == 'S' ) &
                                (f.col('NUM_ANO_COMPETENCIA') == f.year(f.current_date()))
                    ))
    
    df_acordos_promessas = (df_acordos_promessas.join(bi_originais,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI'], "left"))
    
    
    return (df_acordos_promessas
            .drop(bi_originais['DAT_VENCIMENTO'])
            .drop(bi_originais['COD_TITULO'])
           )
           
#--3� TIPO CURSO
#TABELA DUPLICADA !!!! ATENÇÃO 
#COD_CPFCGC_PES
def join_contratos(df_acordos_promessas, bi_contratos):
    df_acordos_promessas = (df_acordos_promessas.join(bi_contratos,
                                  df_acordos_promessas["COD_CONTRATO"] == bi_contratos["COD_CONTRATO"], "left"))


    df_acordos_promessas =  (df_acordos_promessas.withColumn("COD_CPFCGC_PES", f.when(f.col("TIP_MATRICULA") == "PRESENCIAL",
                            f.concat(
                                   f.lpad(f.col("COD_FILIAL"),3,'0'), f.lit(""), f.substring(f.col('COD_CONTRATO_ORIGEM'), 2, 15)
                                    )
                                                                                )
                            .otherwise(
                               f.concat(f.lit("EAD"),f.lit(""), f.col("COD_CONTRATO_ORIGEM"))
                                      )
                                                        )
                        )

    
    return (df_acordos_promessas
            .drop(bi_contratos['COD_CONTRATO'])
            .drop(bi_contratos['DAT_PROCESSAMENTO'])
            .drop(bi_contratos['COD_REGISTRO'])
            .drop(bi_contratos['COD_CLIENTE'])
            .drop(bi_contratos['DSC_SITUACAO_CONTRATO'])
           )
           
#--4� SIUTACAO ALUNO
def join_zeus(df_acordos_promessas, bi_zeus):    
    df_acordos_promessas = (df_acordos_promessas.join(bi_zeus,
                                  df_acordos_promessas['COD_CPFCGC_PES'] == bi_zeus['COD_RA_ZEUS'], "inner"))
    
    return df_acordos_promessas
    

#-- 6� MARCA ALUNOS VERTIA
def join_vertia(df_acordos_promessas, bi_titulos_vertia):    
    df_acordos_promessas = (df_acordos_promessas.join(bi_titulos_vertia,
                                  f.ltrim(f.rtrim(df_acordos_promessas['COD_CONTRATO'])) == f.ltrim(f.rtrim(bi_titulos_vertia['COD_CONTRATO'])), "left"))
      
    df_acordos_promessas = (df_acordos_promessas.withColumn('CARTEIRA', f.lit('VERTIA')))

    
    return (df_acordos_promessas
            .drop(bi_titulos_vertia['COD_CONTRATO'])
            .drop(bi_titulos_vertia['COD_TITULO'])
            .drop(bi_titulos_vertia['DAT_PAGAMENTO'])
            .drop(bi_titulos_vertia['STA_TITULO'])
            .drop(bi_titulos_vertia['COD_CPFCGC_PES'])
           )

    
def execute(spark, inputs, parameters=None, dynamic_parameters=None, write=True):
    bi_originais = inputs['bi_originais']
    bi_acordos_promessas = inputs['bi_acordos_promessas']
    bi_zeus = inputs['base_zeus']
    bi_parcelas = inputs['bi_parcelas']
    bi_contratos = inputs['bi_contrato']
    bi_titulos_vertia = inputs['bi_titulos_vertia']
    
    
    df_acordos_promessas = join_parcelas(bi_acordos_promessas, bi_parcelas)
    df_acordos_promessas = join_originais(df_acordos_promessas, bi_originais)
    df_acordos_promessas = join_contratos(df_acordos_promessas, bi_contratos)
    df_acordos_promessas = join_zeus(df_acordos_promessas, bi_zeus)
    df_acordos_promessas = join_vertia(df_acordos_promessas, bi_titulos_vertia)  
  
  
    #--4� SIUTACAO ALUNO
    df_acordos_promessas = (
        df_acordos_promessas
            .withColumn("SITUACAO_ALUNO", f.when(
                (f.year(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.year(f.current_date())) &
                (f.month(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.month(f.current_date())) 
            ,f.col('STA_OPERACAO_ZEUS')).otherwise(f.lit(None)))
    
            .withColumn("DSC_TIP_CURSO", f.when(
                (f.year(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.year(f.current_date())) &
                (f.month(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.month(f.current_date())) 
            ,f.col('DSC_TIPO_CURSO_ZEUS')).otherwise(f.lit(None)))
    
            .withColumn("TIP_CURSO", f.when(
                (f.year(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.year(f.current_date())) &
                (f.month(df_acordos_promessas['DAT_CRIACAO_PARCELAMENTO']) == f.month(f.current_date())) 
            , f.col('TIP_CURSO_ZEUS')).otherwise(f.lit(None)))
    )
    
  
    df_acordos_promessas =  (df_acordos_promessas.withColumn("EMPAC",
                                f.when(f.substring(f.col("COD_CONTRATO"),1,6) == 'OLIMPO',
                                    f.concat(f.lit("01"),f.lit(""), f.substring(f.col('NUM_ANO_COMPETENCIA'),3,4))
                                )                        
                                .when(f.substring(f.col("COD_CONTRATO"),1,9) == 'COLABORAR', f.concat(f.col('NUM_ANO_COMPETENCIA'),f.lit(""),f.lit("01")))                
                        ))
    
  
    #-- 5� CARTEIRA
    df_acordos_promessas = (
        df_acordos_promessas
        .withColumn("NOM_CARTEIRA", 
            f.when((f.col("TIP_MATRICULA")=='PRESENCIAL') & (f.col('NOM_MARCA_CONTRATO').isin(['ANHANGUERA','UNIDERP'])), 'AEDU PRESENCIAL')
            .when((f.col("TIP_MATRICULA")=='EAD') & (f.col('NOM_MARCA_CONTRATO').isin(['ANHANGUERA','UNIDERP'])), 'AEDU EAD')
            .when((f.col("TIP_MATRICULA")=='PRESENCIAL') & (f.col('NOM_MARCA_CONTRATO').isin(['FAMA','PITAGORAS','UNIC','UNIME','UNOPAR','UNIRONDON'])), 'DM PRESENCIAL')
            .when((f.col("TIP_MATRICULA")=='EAD') & (f.col('NOM_MARCA_CONTRATO').isin(['UNIME','UNOPAR','PITAGORAS','UNIASSELVI','PITAGORAS','FAMA-MACAPA','UNIC','UNIRONDON'])), 'UNOPAR EAD')
    ))
  
    # -- 8� SITUACAO PARCELA Case no campo DS_ST_TITULO criacao do novo campo DSC_SITUACAO_PARCELA
    df_acordos_promessas = (
        df_acordos_promessas
        .withColumn("DSC_SITUACAO_PARCELA", 
            f.when(f.col("VLR_PAGAMENTO").isNotNull(), 'PAGA')
            .when((f.col("VLR_PAGAMENTO").isNull()) & (f.col('DAT_VENCIMENTO') <= f.current_date() - 1), 'EM ATRASO')
            .when((f.col("VLR_PAGAMENTO").isNull()) & (f.col('DAT_VENCIMENTO') >= f.current_date() - 1), 'A VENCER')
            .otherwise(f.col('VLR_PAGAMENTO'))
    ))
    
  
    df_acordos_promessas = (
        df_acordos_promessas
        .withColumn("NUM_DIAS_ATRASO", 
        f.datediff('DAT_VENCIMENTO', 'DAT_CRIACAO_PARCELAMENTO')
    ))
    
    df_acordos_promessas = (
        df_acordos_promessas.select(
            "DSC_TIP_CURSO",
            "SITUACAO_ALUNO",
            "TIP_CURSO",
            "COD_REGISTRO",
            "TIP_TP_MATRICULA",
            "DSC_MARCA_CONTRATO",
            "NOM_NM_CLIENTE",
            "COD_CONTATO",
            "DSC_CPF_CNPJ",
            "COD_PARCELAMENTO",
            "TIP_TP_PARCELAMENTO",
            "NUM_QTDE_PARCELAS",
            "VLR_TOTAL",
            "VLR_DIVIDA",
            "DSC_SCORE_GLOBAL",
            "DSC_SCORE_ENGAJAMENTO",
            "DSC_DESC_CLUSTER",
            "DSC_SITUACAO_CONTRATO",
            "VLR_PRINCIPAL",
            "VLR_DESC_PRINCIPAL",
            "VLR_MULTA",
            "VLR_DESC_MULTA",
            "VLR_JUROS_MORA",
            "VLR_DESC_JUROS_MORA",
            "VLR_JUROS_FINANC",
            "VLR_DESC_JUROS_FINANC",
            "VLR_CORRECAO_MONETARIA",
            "VLR_DESC_CORRECAO_MONETARIA",
            "DSC_AGENCIA_EXTERNA",
            "DSC_GRUPO_USUARIO",
            "DSC_ATENDIMENTO",
            "COD_CONTRATO",
            "DAT_CANCELAMENTO",
            "STA_PARCELAMENTO",
            "NOM_US_CRIACAO_PARCELAMENTO",
            "DAT_CRIACAO_PARCELAMENTO",
            "NUM_PAGAMENTO_ENTRADA",
            "DAT_CRIACAO",
            "NOM_US_CRIACAO",
            "DAT_MODIFICACAO",
            "DSC_US_MODIFICACAO",
            "DSC_NATUREZA_REMATRICULA",
            "IND_FL_STATUS",
            "VLR_TAXA_NEGOCIACAO",
            "VLR_DESC_TAXA_NEGOCIACAO",
            "STA_CANCELAMENTO",
            "COD_CLIENTE",
            "COD_PARCELA",
            "DAT_PROCESSAMENTO",
            "COD_TITULO",
            "NUM_TITULO",
            "DAT_PAGAMENTO",
            "VLR_PAGAMENTO",
            "DAT_VENCIMENTO",
            "VLR_PARCELA",
            "COD_CLIENTE_ORI",
            "NUM_MES_COMPETENCIA",
            "NUM_ANO_COMPETENCIA",
            "NUM_PARCELA",
            "IND_NATUREZA_REMATRICULA",
            "STA_TITULO",
            "DAT_PROCESSAMENTO_ORI",
            "NOM_GRUPO_TELA_REMAT",
            "NUM_CONTADOR_TELA_REMAT",
            "NOM_NATUREZA_TITULO",
            "DSC_NATUREZA_TITULO",
            "VLR_LIQUIDO",
            "TIP_MATRICULA",
            "NOM_MARCA_CONTRATO",
            "NOM_CLIENTE",
            "NOM_SISTEMA_ORIGEM",
            "COD_FILIAL",
            "NUM_CPF_CNPJ",
            "COD_CLIENTE_ORIGEM",
            "COD_CONTRATO_ORIGEM",
            "IND_STATUS",
            "DAT_INSR",
            "COD_CPFCGC_PES",
            "COD_RA_ZEUS",
            "TIP_CURSO_ZEUS",
            "STA_OPERACAO_ZEUS",
            "DSC_TIPO_CURSO_ZEUS",
            "IND_CARTEIRAVERTIA",
            "CARTEIRA",
            "EMPAC",
            "NOM_CARTEIRA",
            "DSC_SITUACAO_PARCELA",
            "NUM_DIAS_ATRASO",
        )
    )


  
    return df_acordos_promessas